In [1]:
import torch                  # main PyTorch library
from torch import nn         # neural network layers
from torch.nn import functional as F  # functions like relu, loss
import math                  # math operations (log, exp, etc.)
import os                    # file handling
import shutil                # file operations (copy, delete)
from sklearn.model_selection import train_test_split  # split dataset

In [18]:
class SelfAttention(nn.Module):
  def __init__(self, n_heads, embd_dim, in_proj_bias=True, out_proj_bias=True):
    super().__init__()
    self.n_heads= n_heads #Number of attention heads / Each head learns different relationships
    self.in_proj=nn.Linear(embd_dim, 3 * embd_dim, bias=in_proj_bias) #creates Q, K, V in one shot
    self.out_proj=nn.Linear(embd_dim ,embd_dim, bias=out_proj_bias)

    self.d_heads=embd_dim // n_heads

  def forward(self, x, causal_mask=False):
    # x = (batch_size, sequence_length, embedding_dimension)
    batch_size, seq_len, embd_dim=x.shape
    # self.in_proj(x)= (batch, seq_len, embd_dim) → 3 *(batch, seq_len, embd_dim)
    interim_shap = (batch_size, seq_len, 3 * embd_dim)
    Q, K, V = self.in_proj(x).chunk(3, dim=-1)

    #change the shape of q, k, v to match the interim_shape
    q = Q.view(batch_size, seq_len, self.n_heads, self.d_heads).transpose(1, 2)
    k = K.view(batch_size, seq_len, self.n_heads, self.d_heads).transpose(1, 2)
    v = V.view(batch_size, seq_len, self.n_heads, self.d_heads).transpose(1, 2)

    #calculate the attention
    weight = q @ k.transpose(-1, -2)

    if causal_mask:
      #mask where the upper triangle (above the principal dagonal) is 1
      mask = torch.ones_like(weight, dtype=torch.bool).triu(1)
      #fill the upper traingle with -inf
      weight.masked_fill(mask == 1, float('-inf'))

    weight = weight / math.sqrt(self.d_heads)
    weight = F.softmax(weight, dim=-1)

    #(batch_size, h_heads, seq_len, dim / h)
    output = weight @ v

    #(batch_size, h_heads, seq_len, dim / h) ->(batch_size, seq_len, h_heads, dim / h)
    output = output.transpose(1, 2)

    #change the shape of the output of out_proj
    output = output.reshape(batch_size, seq_len, embd_dim)

    output = self.out_proj(output)

    return output

In [16]:
class AttentionBlock(nn.Module):
  def __init__(self, channels):
    super().__init__()
    self.groupnorm = nn.GroupNorm(32, channels) #divides the channels into 32 groups and normalizes the features within each group.
    self.attention = SelfAttention(1, channels)

  def forward(self, x):
    #x = (batch_size, channels, h, w)
    residual = x.clone() #taking a "snapshot" of the data before it enters the normalization and attention layers.


    x = self.groupnorm(x)

    n, c, h, w = x.shape
    #(batch_sizen, channels, h * w) -> (batch_sizen, channels, h * w)
    x=x.view(n, c, h*w)
    #(batch_sizen, channels, h * w) -> (batch_sizen,h * w, channels)
    x=x.transpose(-1, -2)

    #perform self_attention without mask
    #(batch_sizen,h * w, channels) -> (batch_sizen,h * w, channels)
    x = self.attention(x)
    #(batch_sizen,h * w, channels) -> (batch_sizen, channels, h * w)
    x=x.transpose(-1, -2)

    #(batch_sizen, channels, h * w) -> (batch_sizen, channels, h, w)
    x=x.view(n, c, h, w)

    x = x + residual #the "overlay" where the original snapshot is added back to the processed version.
    return x

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # First normalization: Groups channels into 32 sets to stabilize training
        self.groupnorm1 = nn.GroupNorm(32, in_channels)

        # First convolution: Extracts spatial features using a 3x3 window
        # Padding=1 ensures the Height and Width of the image don't shrink
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        # Second normalization: Applied after the first feature extraction
        self.groupnorm2 = nn.GroupNorm(32, out_channels)

        # Second convolution: Refines the features within the new channel space
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        # Skip Connection Logic:
        # If input and output channels match, we can add them directly (Identity)
        if in_channels == out_channels:
            self.residual_layer = nn.Identity()
        else:
            # If channels differ, use a 1x1 conv to "project" the input to the output shape
            # so they can be mathematically added together
            self.residual_layer = nn.Conv2d(in_channels, out_channels, kernel_size=1, padding=0)

    def forward(self, x):
        # x shape: (batch_size, channels, height, width)

        # 1. Save a copy of the input to add back at the very end (The "Shortcut")
        residue = x.clone()

        # 2. Start the "Learning Path"
        x = self.groupnorm1(x)

        # SELU activation: Keeps the mean and variance of activations stable
        x = F.selu(x)

        x = self.conv1(x)
        x = self.groupnorm2(x)

        # Note: Usually a second activation (F.selu) goes here before conv2
        x = self.conv2(x)

        # 3. The Residual Step:
        # Add the original input (processed through the residual_layer) back to the output
        # This allows the network to learn only the 'changes' needed, not the whole image
        return x + self.residual_layer(residue)

In [5]:

class Encoder(nn.Sequential):
    def  __init__(self):
        super().__init__(
            # (batch_size, channel, h, w) -> (batch_size, 128, h, w)
            nn.Conv2d(3, 128, kernel_size=3, padding=1),

            # (batch_size, 128, h, w) -> (batch_size, 128, h, w)
            ResidualBlock(128, 128),

            # (batch_size, 128, h, w) -> (batch_size, 128, h / 2, w / 2)
            nn.Conv2d(128, 128, kernel_size=3, stride=2, padding=0),

            # (batch_size, 128, h / 2, w / 2) -> (batch_size, 256, h / 2, w / 2)
            ResidualBlock(128, 256),

            # (batch_size, 256, h / 2, w / 2) -> (batch_size, 256, h / 2, w / 2)
            ResidualBlock(256, 256),

            # (batch_size, 256, h / 2, w / 2) -> (batch_size, 256, h / 4, w / 4)
            nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=0),

            # (batch_size, 256, h / 4, w / 4) -> (batch_size, 512, h / 4, w / 4)
            ResidualBlock(256, 512),

            # (batch_size, 512, h / 4, w / 4) -> (batch_size, 512, h / 4, w / 4)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 4, w / 4) -> (batch_size, 512, h / 8, w / 8)
            nn.Conv2d(512, 512, kernel_size=3, stride=2, padding=0),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            AttentionBlock(512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            ResidualBlock(512, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            nn.GroupNorm(32, 512),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 512, h / 8, w / 8)
            nn.SiLU(),

            # (batch_size, 512, h / 8, w / 8) -> (batch_size, 8, h / 8, w / 8)
            nn.Conv2d(512, 8, kernel_size=3, padding=1),

            # (batch_size, 8, h / 8, w / 8) -> (batch_size, 8, h / 8, w / 8)
            nn.Conv2d(8, 8, kernel_size=1, padding=0)
        )


    def forward(self, x):
    # Loop through every layer defined in the __init__ Sequential block
      for module in self:
        # Check if the layer is a downsampling convolution (stride 2)
        if isinstance(module, nn.Conv2d) and module.stride == (2, 2):
            # Manually add padding to the right and bottom to ensure the
            # math works out perfectly for the stride-2 operation
            x = F.pad(x, (0, 1, 0, 1))

        # Pass the data through the current layer
        x = module(x)

      # After all layers, x is (Batch, 8, H/8, W/8)
      # We split the 8 channels into two groups of 4:
      # 'mean' (the center of our distribution) and 'log_variance' (the spread)
      mean, log_variance = torch.chunk(x, 2, dim=1)

      # Prevent the variance from becoming too extreme (exploding/vanishing)
      log_variance = torch.clamp(log_variance, -30, 20)

      # REPARAMETERIZATION TRICK:
      # 1. Convert log_variance to standard deviation (std)
      std = torch.exp(0.5 * log_variance)
      # 2. Grab a random number from a normal distribution (mean=0, std=1)
      eps = torch.randn_like(std)
      # 3. Create a sample: x = mean + random_noise * spread
      x = mean + eps * std

      # Scale the final result by a specific constant used in Stable Diffusion
      # to keep the latent values in a manageable range for the U-Net.
      x *= 0.18215

      return x


In [6]:

class Decoder(nn.Sequential):
    def __init__(self):
        super().__init__(
            # (batch_size, 4, 32, 32) -> (batch_size, 512, 32, 32)
            nn.Conv2d(4, 512, kernel_size=3, padding=1),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            ResidualBlock(512, 512),

            # (batch_Size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            AttentionBlock(512),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            ResidualBlock(512, 512),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            ResidualBlock(512, 512),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 32, 32)
            ResidualBlock(512, 512),

            # (batch_size, 512, 32, 32) -> (batch_size, 512, 64, 64)
            nn.Upsample(scale_factor=2),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 64, 64)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 64, 64)
            ResidualBlock(512, 512),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 64, 64)
            ResidualBlock(512, 512),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 64, 64)
            ResidualBlock(512, 512),

            # (batch_size, 512, 64, 64) -> (batch_size, 512, 128, 128)
            nn.Upsample(scale_factor=2),

            # (batch_size, 512, 128, 128) -> (batch_size, 512, 128, 128)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),

            # (batch_size, 512, 128, 128) -> (batch_size, 256, 128, 128)
            ResidualBlock(512, 256),

            # (batch_size, 256, 128, 128) -> (batch_size, 256, 128, 128)
            ResidualBlock(256, 256),

            # (batch_size, 256, 128, 128) -> (batch_size, 256, 128, 128)
            ResidualBlock(256, 256),

            # (batch_size, 256, 128, 128) -> (batch_size, 256, 256, 256)
            nn.Upsample(scale_factor=2),

            # (batch_size, 256, 256, 256) -> (batch_size, 256, 256, 256)
            nn.Conv2d(256, 256, kernel_size=3, padding=1),

            # (batch_size, 256, 256, 256) -> (batch_size, 128, 256, 256)
            ResidualBlock(256, 128),

            # (batch_size, 128, 256, 256) -> (batch_size, 128, 256, 256)
            ResidualBlock(128, 128),

            # (batch_size, 128, 256, 256) -> (batch_size, 128, 256, 256)
            ResidualBlock(128, 128),

            nn.GroupNorm(32, 128),

            nn.SiLU(),

            # (batch_size, 128, 256, 256) -> (batch_size, 3, 256, 256)
            nn.Conv2d(128, 3, kernel_size=3, padding=1),
        )
    def forward(self, x):
      # x: (batch_size, 4, h / 8, w / 8) -> The latent representation

      # 1. Reverse the scaling added by the encoder.
      # The encoder multiplied by 0.18215, so we divide here to
      # bring the data back to its original distribution scale.
      x /= 0.18215

      # 2. Sequential Processing
      # Pass the latent vector through all the layers:
      # Conv -> Residual -> Attention -> Upsample -> Conv ...
      for module in self:
          x = module(x)

      # 3. Final Output
      # The result is (batch_size, 3, h, w) - a full-color image.
      return x

In [7]:
!gdown 1KXRTB_q4uub_XOHecpsQjE4Kmv76sZbV

Downloading...
From (original): https://drive.google.com/uc?id=1KXRTB_q4uub_XOHecpsQjE4Kmv76sZbV
From (redirected): https://drive.google.com/uc?id=1KXRTB_q4uub_XOHecpsQjE4Kmv76sZbV&confirm=t&uuid=87697a9b-5186-4276-b3ca-e99f5bc8809b
To: /content/all-dogs.zip
100% 775M/775M [00:10<00:00, 76.8MB/s]


In [8]:
!unzip all-dogs.zip

Streaming output truncated to the last 5000 lines.
  inflating: all-dogs/n02113624_8890.jpg  
  inflating: all-dogs/n02107683_215.jpg  
  inflating: all-dogs/n02093428_5326.jpg  
  inflating: all-dogs/n02105412_8018.jpg  
  inflating: all-dogs/n02093647_3129.jpg  
  inflating: all-dogs/n02088466_8078.jpg  
  inflating: all-dogs/n02108915_4214.jpg  
  inflating: all-dogs/n02089078_2841.jpg  
  inflating: all-dogs/n02105855_3498.jpg  
  inflating: all-dogs/n02094114_2823.jpg  
  inflating: all-dogs/n02088094_649.jpg  
  inflating: all-dogs/n02091831_2232.jpg  
  inflating: all-dogs/n02096585_3105.jpg  
  inflating: all-dogs/n02109961_977.jpg  
  inflating: all-dogs/n02097047_5869.jpg  
  inflating: all-dogs/n02106030_16250.jpg  
  inflating: all-dogs/n02110958_13721.jpg  
  inflating: all-dogs/n02107142_8437.jpg  
  inflating: all-dogs/n02094433_1312.jpg  
  inflating: all-dogs/n02097474_5481.jpg  
  inflating: all-dogs/n02097130_1531.jpg  
  inflating: all-dogs/n02092339_284.jpg  
  inf

In [9]:
def split_dataset(source_dir, train_dir, test_dir, test_size=0.2, random_state=42):
    image_files = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    train_files, test_files = train_test_split(image_files, test_size=test_size, random_state=random_state)

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    for file in train_files:
        shutil.copy(os.path.join(source_dir, file), os.path.join(train_dir, file))

    for file in test_files:
        shutil.copy(os.path.join(source_dir, file), os.path.join(test_dir, file))

    print(f"Dataset split complete. {len(train_files)} training images, {len(test_files)} test images.")

source_dir = "./all-dogs"
train_dir = "./data/train/dogs"
test_dir = "./data/test/dogs"

split_dataset(source_dir, train_dir, test_dir)

Dataset split complete. 16463 training images, 4116 test images.


In [10]:
# Model
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
# from model import Encoder, Decoder

# Device configuration
device = torch.device('cuda')

# Hyperparameters
num_epochs = 100
learning_rate = 1e-4
beta = 0.00025  # KL divergence weight

# Data loading
transform = transforms.Compose([
    transforms.Resize((56, 56)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
batch_size = 4
dataset = torchvision.datasets.ImageFolder(root='./data/train', transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)


model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Add these hyperparameters
accumulation_steps = 1  # Adjust as needed
effective_batch_size = batch_size * accumulation_steps

train_losses = []

# training loop
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for i, (images, _) in enumerate(dataloader):
        images = images.to(device)

        # Forward pass
        reconstructed, encoded = model(images)

        # Compute loss
        recon_loss = nn.MSELoss()(reconstructed, images)

        # Extract mean and log_variance from encoded
        mean, log_variance = torch.chunk(encoded, 2, dim=1)

        kl_div = -0.5 * torch.sum(1 + log_variance - mean.pow(2) - log_variance.exp())
        loss = recon_loss + beta * kl_div

        # Normalize the loss to account for accumulation
        loss = loss / accumulation_steps

        # Backward pass
        loss.backward()

        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        train_loss += loss.item() * accumulation_steps

        print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(dataloader)}], '
              f'Loss: {loss.item()*accumulation_steps:.4f}, Recon Loss: {recon_loss.item():.4f}, KL Div: {kl_div.item():.4f}')



        with torch.no_grad():
            # Take the first image from the batch
            sample_image = images[0].unsqueeze(0)
            sample_reconstructed = model(sample_image)[0]

            sample_image = (sample_image * 0.5) + 0.5
            sample_reconstructed = (sample_reconstructed * 0.5) + 0.5

            torchvision.utils.save_image(sample_reconstructed, 'reconstructed.png')

    train_losses.append(train_loss / len(dataloader))
  # Save the model checkpoint
    torch.save(model.state_dict(), f'vae_model_epoch_{epoch+1}.pth')

print('Training finished!')

Streaming output truncated to the last 5000 lines.
Epoch [12/100], Step [28/4116], Loss: 0.0188, Recon Loss: 0.0185, KL Div: 0.9698
Epoch [12/100], Step [29/4116], Loss: 0.0271, Recon Loss: 0.0268, KL Div: 1.4387
Epoch [12/100], Step [30/4116], Loss: 0.0156, Recon Loss: 0.0153, KL Div: 0.9541
Epoch [12/100], Step [31/4116], Loss: 0.0172, Recon Loss: 0.0167, KL Div: 1.6951
Epoch [12/100], Step [32/4116], Loss: 0.0215, Recon Loss: 0.0208, KL Div: 2.6213
Epoch [12/100], Step [33/4116], Loss: 0.0270, Recon Loss: 0.0265, KL Div: 1.8016
Epoch [12/100], Step [34/4116], Loss: 0.0307, Recon Loss: 0.0302, KL Div: 2.2127
Epoch [12/100], Step [35/4116], Loss: 0.0177, Recon Loss: 0.0174, KL Div: 1.0868
Epoch [12/100], Step [36/4116], Loss: 0.0179, Recon Loss: 0.0175, KL Div: 1.5965
Epoch [12/100], Step [37/4116], Loss: 0.0241, Recon Loss: 0.0239, KL Div: 0.9764
Epoch [12/100], Step [38/4116], Loss: 0.0204, Recon Loss: 0.0202, KL Div: 1.0243
Epoch [12/100], Step [39/4116], Loss: 0.0293, Recon Loss: 

In [ ]:
import matplotlib.pyplot as plt

# plot the loss curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Loss over Time')
plt.legend()
plt.show()